# NLP Assignment 2025/26 - PoliMillionaire Speech Bot

This notebook is a separate speech-mode experiment. It keeps the text benchmark notebook clean and focuses on the audio interface:

1. fetch question and answer-option audio from the official game client;
2. transcribe the audio locally with Whisper;
3. answer with a local open-weight text model;
4. submit the final `option_id` through the official API client.

No external LLM API is used. Whisper and the answer model run locally in Colab.


## 1. Global Settings

All settings for the speech experiment are centralized here.


In [ ]:
# Centralized, all settings are. Scattered through the notebook, they are not.
API_URL = 'http://131.175.15.22:51111/'
USERNAME = 'MarianoAkaMery'
PASSWORD = 'Test1234!'

DRIVE_PROJECT_DIR = '/content/gdrive/MyDrive/[2025-26] - 088946 - NLP PROJECT'

COMPETITION_KEY = 'science_nature'
COMPETITIONS = {
    'entertainment': {'id': 0, 'name': 'Entertainment'},
    'ancient_history_politics': {'id': 1, 'name': 'Ancient History and Politics'},
    'science_nature': {'id': 2, 'name': 'Science and Nature'},
    'maths': {'id': 3, 'name': 'Maths'},
}
COMPETITION_ID = COMPETITIONS[COMPETITION_KEY]['id']
GAME_MODE = 'speech'

ANSWER_MODEL_NAME = 'Qwen/Qwen2.5-3B-Instruct'
WHISPER_MODEL_NAME = 'openai/whisper-small'
LOAD_IN_4BIT = True
MAX_NEW_TOKENS = 2

MAX_LEVELS_TO_PLAY = 15
DELAY_BETWEEN_QUESTIONS_S = 1.0
SAVE_RUN_LOG = True
PRINT_TRANSCRIPTS = True
PRINT_WRONG_QUESTION_DEBUG = True

INSTALL_DEPENDENCIES = True
RUN_API_CHECK = True
RUN_MODEL_WARMUP = True
RUN_FULL_SPEECH_GAME = True

print('Speech settings loaded.')


## 2. Colab Setup

This mounts Google Drive and makes `millionaire_client` importable.


In [ ]:
# Mounted, Google Drive is. Found, the project folder must be.
import os
import sys
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/gdrive/')
    IN_COLAB = True
except Exception:
    IN_COLAB = False

PROJECT_DIR = DRIVE_PROJECT_DIR if IN_COLAB else os.getcwd()
if PROJECT_DIR not in sys.path:
    sys.path.append(PROJECT_DIR)

print('IN_COLAB:', IN_COLAB)
print('PROJECT_DIR:', PROJECT_DIR)
print('millionaire_client exists:', os.path.isdir(os.path.join(PROJECT_DIR, 'millionaire_client')))


## 3. Dependencies

The speech notebook needs local ASR and local text generation packages.


In [ ]:
# Installed, speech and model dependencies are. Locally, everything runs.
if INSTALL_DEPENDENCIES:
    import subprocess
    subprocess.check_call([
        sys.executable,
        '-m',
        'pip',
        'install',
        '-q',
        'transformers',
        'accelerate',
        'sentencepiece',
        'bitsandbytes',
        'torch',
        'soundfile',
    ])
    print('Dependencies installed or already available.')
else:
    print('Dependency installation skipped by settings.')

import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('WARNING: no GPU detected. CPU inference will likely timeout.')


## 4. Official Game Client

Only the course-provided client is used for game interaction.


In [ ]:
# Imported, the official client is. Invented endpoints, used they are not.
from millionaire_client import MillionaireClient, AuthenticationError, MillionaireError, GameError

client = MillionaireClient(API_URL)
user = client.login(USERNAME, PASSWORD)
print(f'Logged in as {user.username} (role={user.role})')

if RUN_API_CHECK:
    competitions = client.competitions.list_all()
    print('Available competitions:')
    for comp in competitions:
        print(f'  {comp.id}: {comp.name} | max_levels={comp.max_levels}')
else:
    print('API check skipped by settings.')


## 5. Category Prompt and Answer Parsing

Whisper transcripts are noisy, so the prompt asks for only one answer letter. The letter is then mapped to the option id delivered by the game state.


In [ ]:
# Built, the prompt is. Parsed, only valid answer letters are.
import re

CATEGORY_PROMPTS = {
    'entertainment': 'You are a careful quiz expert on music, movies, television, celebrities, awards, and pop culture.',
    'ancient_history_politics': 'You are a careful quiz expert on ancient history and politics, especially Rome, Greece, empires, rulers, and institutions.',
    'science_nature': 'You are a careful quiz expert on science and nature, including chemistry, biology, physics, earth science, astronomy, and units.',
    'maths': 'You are a careful mathematics and statistics tutor. Compute exactly and choose the answer choice that matches.',
}

def build_speech_prompt(question_text, option_texts, competition_key=COMPETITION_KEY):
    option_lines = '\n'.join(f'{chr(65 + idx)}) {text}' for idx, text in enumerate(option_texts))
    return f"""{CATEGORY_PROMPTS[competition_key]}

You are playing a multiple-choice quiz from speech transcripts.
The transcript may contain small ASR mistakes.
Choose the single best option.
Return only one letter: A, B, C, or D.
Do not explain.

Question transcript:
{question_text}

Option transcripts:
{option_lines}

Answer:"""

def parse_answer_letter(raw_text, num_options=4):
    valid_letters = [chr(65 + idx) for idx in range(num_options)]
    letter_group = ''.join(valid_letters)
    cleaned = str(raw_text).strip().upper()
    if not cleaned:
        return None
    direct = re.match(rf'^\s*([{letter_group}])(?:\s|\)|\.|:|$)', cleaned)
    if direct:
        return direct.group(1)
    patterns = [
        rf'FINAL\s+ANSWER\s*(?:IS|:)?\s*([{letter_group}])\b',
        rf'ANSWER\s*(?:IS|:)?\s*([{letter_group}])\b',
        rf'OPTION\s*([{letter_group}])\b',
    ]
    for pattern in patterns:
        matches = re.findall(pattern, cleaned)
        if matches:
            return matches[-1]
    standalone = re.findall(rf'(?<![A-Z])([{letter_group}])(?![A-Z])', cleaned)
    return standalone[-1] if standalone else None

def letter_to_option_id(letter, question):
    if letter is None:
        return None
    index = ord(letter) - ord('A')
    if 0 <= index < len(question.options):
        return question.options[index].id
    return None


## 6. Local Whisper ASR

The server provides WAV audio bytes. This cell saves each clip temporarily and transcribes it with a local Whisper pipeline.


In [ ]:
# Transcribed, audio clips are. Locally, Whisper runs.
import tempfile
from transformers import pipeline

asr_pipeline = None

def load_asr():
    global asr_pipeline
    if asr_pipeline is not None:
        return asr_pipeline
    device = 0 if torch.cuda.is_available() else -1
    asr_pipeline = pipeline(
        'automatic-speech-recognition',
        model=WHISPER_MODEL_NAME,
        device=device,
    )
    return asr_pipeline

def transcribe_wav_bytes(audio_bytes, label):
    load_asr()
    with tempfile.NamedTemporaryFile(suffix=f'_{label}.wav', delete=False) as tmp:
        tmp.write(audio_bytes)
        tmp_path = tmp.name
    try:
        result = asr_pipeline(tmp_path)
        return str(result.get('text', '')).strip()
    finally:
        try:
            os.remove(tmp_path)
        except OSError:
            pass


## 7. Local Answer Model

This is the same type of local answer model used by the text notebook.


In [ ]:
# Loaded locally, the answer model is. External LLM API, used it is not.
import time

class LocalCausalLMAnswerer:
    def __init__(self, model_name, max_new_tokens=8):
        self.model_name = model_name
        self.max_new_tokens = max_new_tokens
        self.tokenizer = None
        self.model = None

    def load(self):
        from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
        if self.model is not None:
            return

        self.tokenizer = AutoTokenizer.from_pretrained(self.model_name)
        quantization_config = None
        if LOAD_IN_4BIT:
            quantization_config = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_use_double_quant=True,
                bnb_4bit_quant_type='nf4',
                bnb_4bit_compute_dtype=torch.float16,
            )

        self.model = AutoModelForCausalLM.from_pretrained(
            self.model_name,
            device_map={'': 0} if torch.cuda.is_available() else 'cpu',
            torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
            quantization_config=quantization_config,
        )
        self.model.eval()

    def answer(self, question_text, option_texts, question_state, competition_key=COMPETITION_KEY):
        self.load()
        prompt = build_speech_prompt(question_text, option_texts, competition_key)
        inputs = self.tokenizer(prompt, return_tensors='pt').to(self.model.device)

        start = time.time()
        output_ids = self.model.generate(
            **inputs,
            max_new_tokens=self.max_new_tokens,
            do_sample=False,
            pad_token_id=self.tokenizer.eos_token_id,
        )
        elapsed_s = time.time() - start

        generated_ids = output_ids[0][inputs['input_ids'].shape[-1]:]
        raw = self.tokenizer.decode(generated_ids, skip_special_tokens=True).strip()
        letter = parse_answer_letter(raw, num_options=len(option_texts))
        option_id = letter_to_option_id(letter, question_state)

        return {
            'model': self.model_name,
            'raw': raw,
            'letter': letter,
            'option_id': option_id,
            'elapsed_s': elapsed_s,
        }

answer_model = LocalCausalLMAnswerer(ANSWER_MODEL_NAME, max_new_tokens=MAX_NEW_TOKENS)
print('Answer model:', ANSWER_MODEL_NAME)


## 8. Warm-up

Models are loaded before a real speech game starts. In speech mode the timer starts after option D is requested, so model loading during the game would be fatal.


In [ ]:
# Warmed up, ASR and answer model are. During the game, loaded they should not be.
import gc

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

if RUN_MODEL_WARMUP:
    print('Loading Whisper ASR...')
    load_asr()
    print('Whisper ready.')

    print('Loading answer model...')
    answer_model.load()
    print('Answer model ready.')

    print('Warm-up complete.')
else:
    print('Warm-up skipped by settings.')


## 9. Full Speech Game

This starts one real speech-mode game. Audio clips are transcribed first, then the answer model chooses the option id.


In [ ]:
# Played, a full speech game is. Transcribed, each audio option becomes.
import json
from datetime import UTC, datetime

OPTION_LABELS = ['A', 'B', 'C', 'D']

def fetch_and_transcribe_current_question(game):
    question_audio = game.fetch_audio_question()
    question_text = transcribe_wav_bytes(question_audio, f'level_{game.current_level}_question')

    option_texts = []
    option_audio_lengths = []
    for option_index in range(4):
        option_audio = game.fetch_audio_option_next()
        option_audio_lengths.append(len(option_audio))
        option_text = transcribe_wav_bytes(option_audio, f'level_{game.current_level}_option_{OPTION_LABELS[option_index]}')
        option_texts.append(option_text)

    game.refresh_state()
    return question_text, option_texts, len(question_audio), option_audio_lengths

def play_full_speech_game(competition_key=COMPETITION_KEY):
    run_log = []
    competition_id = COMPETITIONS[competition_key]['id']
    game = client.game.start(competition_id=competition_id, mode='speech')
    print(f'Started speech session: {game.session_id}')

    while game.in_progress and game.current_level <= MAX_LEVELS_TO_PLAY:
        question_state = game.current_question
        if question_state is None:
            break

        question_text, option_texts, question_audio_len, option_audio_lens = fetch_and_transcribe_current_question(game)
        time_before_s = game.time_remaining

        if PRINT_TRANSCRIPTS:
            print(f'\nLevel {game.current_level} transcripts')
            print('Q:', question_text)
            for idx, opt_text in enumerate(option_texts):
                print(f'  {OPTION_LABELS[idx]}) {opt_text}')
            print(f'Time remaining after audio+ASR: {time_before_s:.1f}s')

        decision = answer_model.answer(question_text, option_texts, question_state, competition_key)
        chosen_id = decision['option_id'] if decision['option_id'] is not None else question_state.options[0].id
        chosen_letter = decision['letter'] if decision['letter'] is not None else 'A'
        result = game.answer(chosen_id)

        print(
            f"Level {game.current_level} | "
            f"chosen={chosen_letter}:{chosen_id} | "
            f"correct={result.correct} | "
            f"timeout={result.timed_out} | "
            f"model_time={decision['elapsed_s']:.2f}s | "
            f"time_before_answer={time_before_s:.1f}s | "
            f"earned={result.earned_amount}"
        )

        if PRINT_WRONG_QUESTION_DEBUG and result.correct is False:
            print('  Wrong speech-question debug:')
            print(f'    transcript Q: {question_text}')
            for idx, opt_text in enumerate(option_texts):
                print(f'    {OPTION_LABELS[idx]}) id={question_state.options[idx].id} | transcript={opt_text}')
            print(f"    raw={decision['raw']!r}")

        run_log.append({
            'timestamp': datetime.now(UTC).isoformat(),
            'session_id': game.session_id,
            'competition_key': competition_key,
            'competition_name': COMPETITIONS[competition_key]['name'],
            'mode': 'speech',
            'level': game.current_level,
            'question_id': question_state.id,
            'question_transcript': question_text,
            'option_transcripts': option_texts,
            'question_audio_bytes': question_audio_len,
            'option_audio_bytes': option_audio_lens,
            'time_remaining_before_answer_s': time_before_s,
            'chosen_option_id': chosen_id,
            'chosen_letter': chosen_letter,
            'decision': decision,
            'correct': result.correct,
            'timed_out': result.timed_out,
            'game_over': result.game_over,
            'earned_amount': result.earned_amount,
        })

        if result.game_over or result.timed_out:
            break

        time.sleep(DELAY_BETWEEN_QUESTIONS_S)

    print(f'Final level: {game.current_level}')
    print(f'Final earned amount: {game.earned_amount}')
    return game, run_log

if RUN_FULL_SPEECH_GAME:
    final_game, speech_run_log = play_full_speech_game(COMPETITION_KEY)
else:
    final_game, speech_run_log = None, []
    print('Full speech game skipped by settings.')


## 10. Save Speech Results

The speech log includes transcripts, timings, model outputs, and final outcomes.


In [ ]:
# Saved, the speech run is. Analyzed later, ASR errors can be.
if SAVE_RUN_LOG and speech_run_log:
    log_dir = Path(PROJECT_DIR) / 'runs'
    log_dir.mkdir(parents=True, exist_ok=True)
    output_path = log_dir / f'speech_run_{datetime.now(UTC).strftime("%Y%m%d_%H%M%S")}.json'
    with output_path.open('w', encoding='utf-8') as f:
        json.dump(speech_run_log, f, indent=2, ensure_ascii=False)
    print('Saved speech run log:', output_path)
else:
    print('No speech run log saved.')


## 11. Speech Leaderboard

Speech mode has a separate leaderboard query.


In [ ]:
# Checked, the speech leaderboard is. Compared, our speech run can be.
leaderboard = client.leaderboard.get(competition_id=COMPETITION_ID, limit=20, mode='speech')
print('Speech leaderboard:', leaderboard.competition.name)
for rank, entry in enumerate(leaderboard.entries, 1):
    marker = ' <-- us' if entry.username == USERNAME else ''
    print(f'{rank}. {entry.username}: {entry.score} | level={entry.reached_level}{marker}')


## 12. Notes

Speech mode is expected to be harder than text mode because audio fetching, ASR latency, and transcription errors all happen before the final answer. The main quantities to analyze are transcript quality, time remaining after ASR, and whether wrong answers come from ASR errors or model reasoning errors.
